## Extração

In [ ]:
# Importando bibliotecas
import pandas as pd
import requests
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [47]:
# Pegando a última versão disponível
URL_VERSION = "https://ddragon.leagueoflegends.com/api/versions.json"
response = requests.get(URL_VERSION)
versions = response.json()
last_version = versions[0]

# Trazendo dados de todos os campeões com base na última versão disponível
URL_CHAMPIONS = f"https://ddragon.leagueoflegends.com/cdn/{last_version}/data/pt_BR/champion.json"
response = requests.get(URL_CHAMPIONS)
champions = response.json()

## Transformação

In [48]:
# Trazendo somente o dicionário interno 'data'
data_champions = champions['data']

# Normalizando os dados json para um dataframe
df = pd.json_normalize(list(data_champions.values()))

df = df.drop(columns=['version','key','id','image.x','image.y','image.w','image.h','image.group','image.sprite'])

# Alterando nome das colunas
df.columns = [c.replace('.', '_') for c in df.columns]

# Mapeamento de Tags (Categorias de Campeão)
mapa_tags = {
    "Fighter": "Lutador",
    "Assassin": "Assassino",
    "Mage": "Mago",
    "Tank": "Tanque",
    "Marksman": "Atirador",
    "Support": "Suporte"
}

df['tags'] = df['tags'].apply(
    lambda lista_tags: [mapa_tags.get(tag, tag) for tag in lista_tags]
)

df['tags'] = df['tags'].apply(lambda x: ', '.join(x))

mapa_colunas = {
    # Informações Gerais
    "name": "nome",
    "title": "titulo",
    "blurb": "descricao",
    "tags": "tags",
    "partype": "tipo_recurso",
    "image_full": "imagem_nome",
    
    # Info de Gameplay
    "info_attack": "ataque_nivel",
    "info_defense": "defesa_nivel",
    "info_magic": "magia_nivel",
    "info_difficulty": "dificuldade_nivel",
    
    # Stats Base
    "stats_hp": "vida_base",
    "stats_hpperlevel": "vida_por_nivel",
    "stats_mp": "recurso_base",
    "stats_mpperlevel": "recurso_por_nivel",
    "stats_movespeed": "velocidade_movimento",
    "stats_armor": "armadura_base",
    "stats_armorperlevel": "armadura_por_nivel",
    "stats_spellblock": "resistencia_magica_base",
    "stats_spellblockperlevel": "resistencia_magica_por_nivel",
    "stats_attackrange": "alcance_ataque",
    
    # Regeneração
    "stats_hpregen": "regeneracao_vida",
    "stats_hpregenperlevel": "regeneracao_vida_por_nivel",
    "stats_mpregen": "regeneracao_recurso",
    "stats_mpregenperlevel": "regeneracao_recurso_por_nivel",
    
    # Ataque e Crítico
    "stats_crit": "critico_base",
    "stats_critperlevel": "critico_por_nivel",
    "stats_attackdamage": "dano_ataque_base",
    "stats_attackdamageperlevel": "dano_ataque_por_nivel",
    "stats_attackspeedperlevel": "velocidade_ataque_por_nivel",
    "stats_attackspeed": "velocidade_ataque_base"
}

# Aplicando a renomeação no seu DataFrame
df = df.rename(columns=mapa_colunas)

# Carregamento

In [ ]:
load_dotenv()

# Configuração da String de Conexão
usuario = os.getenv("DB_USER")
senha   = os.getenv("DB_SENHA")
host    = os.getenv("DB_HOST")
porta   = os.getenv("DB_PORT")
banco   = os.getenv("DB_NAME")

# String de conexão
connection_str = f"mysql+pymysql://{usuario}:{senha}@{host}:{porta}/{banco}"

# Motor que faz conexão com o banco
engine = create_engine(connection_str)

# Criando tabela no banco de dados com dados do dataframe pandas
df.to_sql(name='campeoes', con=engine, if_exists='replace', index=False)

172